In [1]:
import json
import logging
from dataclasses import asdict

import optuna
import pandas as pd
from IPython.display import display

from elo_model.analysis import fingerprint
from elo_model.config import PipelineConfig
from elo_model.evaluation import BAND_NAMES, rating_bands
from elo_model.feature_schema import FEATURE_NAMES
from elo_model.pipeline import (
    analyze_dataset,
    export_sample,
    extract_dataset,
    feature_missingness,
    load_feature_metadata,
    manifest_rows,
    save_feature_dataset,
    split_summary,
)
from elo_model.predictor import RatingPredictor, export_bundle
from elo_model.split import PlayerSplitter
from elo_model.storage import write_json, write_parquet
from elo_model.training import ModelTrainer
from training_data import ChessComConnection, ModelDatasetBuilder

settings = PipelineConfig()

logging.basicConfig(
    level=settings.collection.log_level,
    format=settings.collection.log_format,
    force=True,
)
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f"Target: {settings.target_games:,} unique games before splitting")
print(f"Existing usernames: {settings.usernames_path.resolve()}")
print(f"Persistent engine cache: {settings.engine_cache_directory.resolve()}")

C:\Users\mc678p\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Target: 1,000 unique games before splitting
Existing usernames: C:\Users\mc678p\OneDrive - AT&T Services, Inc\Documents\Tower Strategy & Roaming\Analysis\ELO Estimator\player_usernames.json
Persistent engine cache: C:\Users\mc678p\OneDrive - AT&T Services, Inc\Documents\Tower Strategy & Roaming\Analysis\ELO Estimator\data\analysis_cache


In [2]:
saved_usernames = json.loads(settings.usernames_path.read_text(encoding="utf-8"))

if not isinstance(saved_usernames, list) or not all(isinstance(name, str) for name in saved_usernames):
    raise ValueError("The username file must contain a JSON list of strings")

usernames = list(dict.fromkeys(
    name.strip().casefold()
    for name in saved_usernames
    if name.strip()
))

if not usernames:
    raise ValueError("The saved username list is empty")

# A seeded shuffle avoids favoring the beginning of the existing username list.
usernames = pd.Series(usernames).sample(
    frac=1,
    random_state=settings.training.seed,
).tolist()

connection = ChessComConnection(settings.collection)
builder = ModelDatasetBuilder(connection)
settings.data_directory.mkdir(parents=True, exist_ok=True)

collection_settings = {
    "usernames_hash": fingerprint(usernames),
    "months": [list(month) for month in settings.collection.months],
    "source_player_cap": settings.collection.max_games_per_player,
}

if settings.collection_settings_path.exists():
    previous_settings = json.loads(settings.collection_settings_path.read_text(encoding="utf-8"))

    if previous_settings != collection_settings:
        raise ValueError(
            "Collection settings changed. Restore them or choose another data_directory; "
            "keep engine_cache_directory unchanged to reuse evaluations."
        )
elif settings.raw_games_path.exists():
    raise ValueError("Cached games have no collection provenance; choose another data_directory")

write_json(settings.collection_settings_path, collection_settings)

print(f"Loaded {len(usernames):,} existing usernames; no player search will run")

Loaded 1,000 existing usernames; no player search will run


In [3]:
cached_games = None

if settings.raw_games_path.exists():
    cached_games = pd.read_parquet(settings.raw_games_path)
    print(f"Reusing {len(cached_games):,} previously collected games")

games_df = builder.build(
    usernames,
    target_games=settings.target_games,
    initial_games=cached_games,
    progress=settings.training.progress,
    checkpoint=lambda frame: write_parquet(settings.raw_games_path, frame),
)

write_parquet(settings.raw_games_path, games_df)

if len(games_df) < settings.target_games:
    raise ValueError(
        f"Only {len(games_df):,}/{settings.target_games:,} eligible games found. "
        "Add archive months or raise the source-player cap, using a new data_directory. "
        "Keep engine_cache_directory to reuse analysis."
    )

print(f"Ready: {len(games_df):,} unique clocked games")
print("Rejected games:", dict(builder.rejections))

Reusing 1,000 previously collected games
Ready: 1,000 unique clocked games
Rejected games: {}


In [4]:
ratings = pd.concat(
    [games_df["white_rating"], games_df["black_rating"]],
    ignore_index=True,
)

rating_coverage = pd.Series(rating_bands(ratings)).value_counts(sort=False)
category_coverage = games_df["category"].value_counts()

print("Collected game categories")
display(category_coverage.rename("games").to_frame())

print("Player-side rating coverage")
display(rating_coverage.rename("rows").to_frame())

display(games_df.head())

Collected game categories


,games
category,
blitz,877
bullet,70
rapid,53


Player-side rating coverage


,rows
<1000,826
1000-1400,130
1400-1800,430
1800-2200,280
2200+,334


,game_id,pgn,white_username,black_username,white_rating,black_rating,category,termination,time_control
0,8f9c81dd03f7be38bb5b5d77f906d7e9f009216bcd4527...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat...",Mploki06,gangdang,1452,1423,blitz,resignation,300
1,975cc5c8153c4d7ae22c78c0f751509c77f40833b3f698...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat...",Sqshimi,gangdang,1414,1431,blitz,timeout,300
2,d6d715c88e01301826d76502b4ca5cf6036909f199497e...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat...",gangdang,esketet1,1423,1429,blitz,timeout,300
3,52bc63b6ab03851190381398da4e0e7138a8aa4c3856ea...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat...",gangdang,zxc0956,1414,1418,blitz,checkmate,300
4,1d69f24e0ff2086d328a101b0232fc2cd7cb93aad31ac8...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat...",WhyHeyman,gangdang,1433,1406,blitz,resignation,300


In [5]:
splitter = PlayerSplitter(
    fractions=settings.split_fractions,
    seed=settings.training.seed,
    strict_participants=settings.strict_participants,
)

split_games = splitter.assign(games_df)
coverage = pd.DataFrame.from_dict(split_summary(split_games), orient="index")

display(coverage)

# Reject unusable hold-outs before paying for engine analysis.
insufficient_rows = coverage["rows"] < settings.training.min_rows
insufficient_players = coverage["target_players"] < settings.min_players_per_partition

if insufficient_rows.any() or insufficient_players.any():
    raise ValueError(
        "Split coverage is too small. Increase target_games or revise split_fractions "
        "before running Stockfish."
    )

write_parquet(settings.split_path, split_games)

side_rows = manifest_rows(split_games)
side_rows["rating_band"] = rating_bands(side_rows["rating"])

display(pd.crosstab(side_rows["partition"], side_rows["rating_band"], dropna=False))
display(pd.crosstab(side_rows["partition"], side_rows["category"]))

,rows,games,target_players
train,1464,821,781
validation,96,90,54
test,96,89,60


rating_band,<1000,1000-1400,1400-1800,1800-2200,2200+
partition,,,,,
test,47,5,11,7,26
train,585,87,335,232,225
validation,41,9,22,5,19


category,blitz,bullet,rapid
partition,,,
test,87,7,2
train,1283,100,81
validation,79,9,8


In [6]:
# Cache keys include game history, engine binary/version and search settings.
engine_metadata = analyze_dataset(
    split_games,
    config=settings.engine,
    cache_directory=settings.engine_cache_directory,
    output=settings.analyzed_path,
    progress=settings.training.progress,
)

analyzed_games = pd.read_parquet(settings.analyzed_path)

print(f"Analysis references saved for {len(analyzed_games):,} games")
print("Rerunning this cell reuses complete cached evaluations")

Analyze games: 100%|██████████| 1000/1000 [1:03:38<00:00,  3.82s/game, cached=14, new=986]


Analysis references saved for 1,000 games
Rerunning this cell reuses complete cached evaluations


In [7]:
# Feature thresholds can change without rerunning the cached engine searches.
features_df = extract_dataset(
    analyzed_games,
    cache_directory=settings.engine_cache_directory,
    config=settings.features,
    progress=settings.training.progress,
)

save_feature_dataset(
    features_df,
    output=settings.features_path,
    engine_metadata=engine_metadata,
    config=settings.features,
)

missingness = pd.Series(feature_missingness(features_df), name="missing_fraction")

display(missingness.sort_values(ascending=False).to_frame())
print(f"Saved {len(features_df):,} player rows with {len(FEATURE_NAMES)} features")

Extract features:   0%|          | 0/1000 [00:00<?, ?game/s]

Extract features: 100%|██████████| 1000/1000 [05:31<00:00,  3.02game/s]


,missing_fraction
self_time_trouble_blunder_rate,0.756643
opponent_time_trouble_blunder_rate,0.756039
self_endgame_top1_rate,0.445048
self_endgame_blunder_rate,0.445048
self_endgame_loss_mean,0.445048
...,...
opponent_think_total,0.000000
opponent_time_trouble_share,0.000000
opponent_clock_budget_used,0.000000
opponent_instant_rate,0.000000


Saved 1,656 player rows with 109 features


In [8]:
feature_metadata = load_feature_metadata(settings.features_path)
training_rows = pd.read_parquet(settings.features_path)

trainer = ModelTrainer(settings.training)
result = trainer.fit(training_rows)

# Preserve the exact feature and engine contract alongside the fitted trees.
result.metadata.update(feature_metadata)
result.metadata["dataset_games"] = len(games_df)
result.metadata["split_fractions"] = list(settings.split_fractions)
result.metadata["strict_participants"] = settings.strict_participants

print("Training complete; evaluation used untouched test rows after selection")

Train models: 100%|██████████| 1/1 [01:10<00:00, 70.45s/model]

Training complete; evaluation used untouched test rows after selection


In [9]:
metric_rows = []

for model_name, report in result.report.items():
    for partition, estimators in report["partitions"].items():
        for estimator_name, metrics in estimators.items():
            metric_rows.append({
                "model": model_name,
                "partition": partition,
                "estimator": estimator_name,
                **metrics["overall"],
            })

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)

for model_name, report in result.report.items():
    test_metrics = report["partitions"]["test"]["lightgbm"]

    print(f"{model_name}: test errors by rating band")
    display(pd.DataFrame.from_dict(test_metrics["by_rating_band"], orient="index"))

    print(f"{model_name}: test errors by time category")
    display(pd.DataFrame.from_dict(test_metrics["by_category"], orient="index"))

    if "feature_importance" in report:
        importance = pd.DataFrame.from_dict(report["feature_importance"], orient="index")
        display(importance.sort_values("gain", ascending=False))

sample = export_sample(training_rows, result.models)
verification = export_bundle(
    result.models,
    directory=settings.artifact_directory,
    sample=sample,
    metadata=result.metadata,
    report=result.report,
    feature_config=settings.features,
)

print("Export verification:")
display(pd.Series(verification).to_frame("value"))
print(f"Model bundle: {settings.artifact_directory.resolve()}")

,model,partition,estimator,rows,players,mae,rmse,player_macro_mae
0,pooled,train,lightgbm,1464,781,217.700880,345.922732,224.840384
1,pooled,train,ridge,1464,781,375.859237,470.508209,386.523577
2,pooled,train,train_median,1464,781,632.454918,736.965909,637.953537
3,pooled,validation,lightgbm,96,54,338.565216,439.799419,320.905824
4,pooled,validation,ridge,96,54,366.980783,463.664941,381.073482
5,pooled,validation,train_median,96,54,634.739583,741.568250,679.495370
6,pooled,test,lightgbm,96,60,427.796413,535.369027,377.759781
7,pooled,test,ridge,96,60,442.294024,549.650464,388.599470
8,pooled,test,train_median,96,60,736.031250,802.783303,716.031667


pooled: test errors by rating band


,rows,players,mae,rmse,player_macro_mae
<1000,47,29,560.346186,647.318983,491.057083
1000-1400,5,3,380.334202,409.530331,340.204269
1400-1800,11,11,207.480830,302.182173,207.480830
1800-2200,7,5,338.736242,433.178417,396.628383
2200+,26,13,314.502736,423.581654,259.651068


pooled: test errors by time category


,rows,players,mae,rmse,player_macro_mae
blitz,87,56,419.262873,530.196593,374.671291
bullet,7,4,579.614512,639.422556,470.226887
rapid,2,2,267.642092,313.893181,267.642092


,gain,gain_fraction,splits
self_phase,11429.880113,0.122135,850
self_accuracy,11301.250131,0.120761,881
opponent_accuracy,10888.605390,0.116351,856
opponent_phase,9832.243476,0.105064,729
opponent_clock,9712.984042,0.103789,768
self_clock,8815.727911,0.094202,696
self_position,5897.091068,0.063014,344
self_complexity,5467.184678,0.058420,416
opponent_complexity,5135.615831,0.054877,397
opponent_position,4801.712660,0.051309,313


Export verification:


,value
sample_rows,64.000000
max_absolute_difference,0.000000
warm_single_row_ms,1.471783
native_model_bytes,871030.000000


Model bundle: C:\Users\mc678p\OneDrive - AT&T Services, Inc\Documents\Tower Strategy & Roaming\Analysis\ELO Estimator\artifacts\rating_model


In [10]:
predictor = RatingPredictor(settings.artifact_directory)

held_out = training_rows.loc[training_rows["partition"] == "test"].head(
    settings.preview_rows
)

predictions = predictor.predict(
    held_out.loc[:, list(FEATURE_NAMES)],
    categories=held_out["category"].tolist(),
)

prediction_preview = held_out.loc[:, ["target_username", "category", "rating"]].copy()
prediction_preview["predicted_rating"] = predictions
prediction_preview["absolute_error"] = (
    prediction_preview["predicted_rating"] - prediction_preview["rating"]
).abs()

display(prediction_preview)

print("Inference used the exported native model; no engine analysis was rerun")

,target_username,category,rating,predicted_rating,absolute_error
21,the-angel-tribe,blitz,2201,2349.477377,148.477377
55,riky366,blitz,1664,1811.760199,147.760199
63,paapaa7,blitz,1773,1791.127579,18.127579
92,zojosh9,blitz,1690,1149.412069,540.587931
110,incognito6868,blitz,1639,1808.720296,169.720296
140,shbahodir,blitz,2323,2422.474933,99.474933
142,felix-rose,blitz,2274,2456.276222,182.276222
190,broadsky21,blitz,228,970.134630,742.134630
193,kannanchesspoy,blitz,293,884.539522,591.539522
205,rolletta,blitz,1818,1429.294317,388.705683


Inference used the exported native model; no engine analysis was rerun


In [20]:
from dataclasses import replace
from datetime import datetime, timedelta, timezone

from elo_model.analysis import AnalysisCache, StockfishAnalyzer
from elo_model.game import GameInput

username = "MCodr"
game_index = 0

current_month = datetime.now(timezone.utc).date().replace(day=1)
previous_month = current_month - timedelta(days=1)

recent_settings = replace(
    settings.collection,
    months=(
        (current_month.year, 9),
        (previous_month.year, 8),
    ),
    max_games_per_player=10,
)

recent_games = ModelDatasetBuilder(
    ChessComConnection(recent_settings)
).build([username], target_games=10)

if not 0 <= game_index < len(recent_games):
    raise ValueError(f"Choose an index from {len(recent_games)} eligible games")

selected = recent_games.iloc[game_index+2]
game = GameInput.from_pgn(selected["pgn"])

with StockfishAnalyzer(
    settings.engine,
    AnalysisCache(settings.engine_cache_directory),
) as analyzer:
    _, analysis = analyzer.analyze(game, progress=True)
    estimates = predictor.predict_game(
        game, analysis, engine_metadata=analyzer.metadata
    )

comparison = pd.DataFrame([
    {
        "player": selected[f"{color}_username"],
        "color": color,
        "actual_rating": selected[f"{color}_rating"],
        "estimated_rating": round(estimates[color]),
    }
    for color in ("white", "black")
])
comparison["absolute_error"] = (
    comparison["estimated_rating"] - comparison["actual_rating"]
).abs()

print(game.headers.get("Date"), game.headers.get("Result"))
display(comparison)

2026.09.27 0-1


,player,color,actual_rating,estimated_rating,absolute_error
0,Neomaxp,white,1351,1439,88
1,MCodr,black,1372,1418,46
